# Pipelines

In [31]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import AmlCompute
from azure.identity import InteractiveBrowserCredential

from azure.ai.ml import load_component, Input
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.constants import AssetTypes


from dotenv import load_dotenv
import os
import yaml
from pathlib import Path

load_dotenv("../.env", override=True)


True

In [24]:
subscription_id = os.environ["ARM_SUBSCRIPTION_ID"]
resource_group = os.environ["ARM_RESOURCE_GROUP"]
workspace = os.environ["AZ_ML_NAME"]
mlflow_tracking_uri = os.environ["MLFLOW_TRACKING_URI"]

credential = InteractiveBrowserCredential(tenant_id=os.environ["ARM_TENANT_ID"])

# Request an MFA-backed token up front; MLClient then reuses it from the cache
credential.get_token(
    "https://management.azure.com/.default",
    claims='{"access_token":{"amr":{"essential":true,"values":["mfa"]}}}',
)

ml_client = MLClient(credential, subscription_id, resource_group, workspace)

/home/dgouwy/projects/mlops-azure/.venv/lib/python3.12/site-packages/msal/oauth2cli/oauth2.py:461: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(
gio: https://login.microsoftonline.com/d018aec4-2b2b-4c66-9939-2c96877e6bf1/oauth2/v2.0/authorize?client_id=04b07795-8ddb-461a-bbee-02f9e1bf7b46&response_type=code&redirect_uri=http%3A%2F%2Flocalhost%3A8400&scope=https%3A%2F%2Fmanagement.azure.com%2F.default+offline_access+openid+profile&state=xg6EpMJHGTlup9L6lgPOIA&code_challenge=SFXNcnDfrS7uT1v2iyl7Mb7BJOifBl3d_CLBTV3R9hM&code_challenge_method=S256&nonce=01d79ad5bc8e354c41db1aa8a5719f87a2671da6fa9dbafa9945192029c581b7&client_info=1&prompt=select_account&claims=%7B%22access_token%22%3A%7B%22amr%22%3A%7B%22essential%22%3Atrue%2C%22values%22%3A%5B%22mfa%22%5D%7D%7D%7D: Operation not supported
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not al

# Create components

In [30]:
parent_dir = Path("../mlops/pipelines/components")
loaded_component_prep = load_component(source=parent_dir / "preprocessing.yml")

prep = ml_client.components.create_or_update(loaded_component_prep)

In [46]:
parent_dir = Path("../mlops/pipelines/components")
loaded_component_training = load_component(source=parent_dir / "training.yml")

training = ml_client.components.create_or_update(loaded_component_training)

# Create Pipelines

In [47]:
# Create the yaml with the pipeline decorator

@pipeline()
def pipeline_function_name(pipeline_job_input):
    preprocessing = loaded_component_prep(input_data=pipeline_job_input)
    training = loaded_component_training(
        training_data=preprocessing.outputs.training_data,
        validation_data=preprocessing.outputs.validation_data,
        n_estimators=100,
        max_depth=3,
        learning_rate=0.3
        )

    return {
        "pipeline_job_training_data": preprocessing.outputs.training_data,
        "pipeline_job_validation_data": preprocessing.outputs.validation_data,
        "pipeline_job_trained_model": training.outputs.model_output,
    }

# Add the input to the pipeline job
pipeline_job = pipeline_function_name(
    Input(type=AssetTypes.URI_FILE, 
    path="azureml:diabetes-csv:1"
))



In [48]:
print(pipeline_job)

display_name: pipeline_function_name
type: pipeline
inputs:
  pipeline_job_input:
    type: uri_file
    path: azureml:diabetes-csv:1
outputs:
  pipeline_job_training_data:
    type: uri_folder
  pipeline_job_validation_data:
    type: uri_folder
  pipeline_job_trained_model:
    type: mlflow_model
jobs:
  preprocessing:
    type: command
    inputs:
      input_data:
        path: ${{parent.inputs.pipeline_job_input}}
    outputs:
      training_data: ${{parent.outputs.pipeline_job_training_data}}
      validation_data: ${{parent.outputs.pipeline_job_validation_data}}
    component:
      $schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
      name: preprocessing
      version: '2'
      display_name: Preprocess data
      description: Splits diabetes data into train/validation sets and scales the
        features.
      type: command
      inputs:
        input_data:
          type: uri_file
        valid_size:
          type: number
          default:

In [ ]:
# Change the yaml file

# change the output mode
pipeline_job.outputs.pipeline_job_training_data.mode = "upload"
pipeline_job.outputs.pipeline_job_validation_data.mode = "upload"
pipeline_job.outputs.pipeline_job_model_output.mode = "upload"

# set pipeline level compute
pipeline_job.settings.default_compute = "cpu-training-cluster"

# set pipeline level datastore
pipeline_job.settings.default_datastore = "workspaceblobstore"

In [50]:
# Dump the yaml file to the mlops/pipelines folder
pipeline_job.dump(Path("..") / "mlops" / "pipelines" / "pipeline.yml")

In [51]:
# submit job to workspace
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name="diabetes_pipeline"
)

# Schedule a pipeline

In [ ]:
from azure.ai.ml.entities import RecurrenceTrigger
from azure.ai.ml.entities import JobSchedule

In [ ]:
schedule_name = "run_every_ten_minutes"

# First create a recurrence trigger
recurrence_trigger = RecurrenceTrigger(
    frequency="minute",
    interval=10,
)

# Create the schedule
job_schedule = JobSchedule(
    name=schedule_name, trigger=recurrence_trigger, create_job=pipeline_job
)

# Create it all!
job_schedule = ml_client.schedules.begin_create_or_update(
    schedule=job_schedule
).result()

In [ ]:
# First disable schedule
ml_client.schedules.begin_disable(name=schedule_name).result()

# Then delete it
ml_client.schedules.begin_delete(name=schedule_name).result()